# MT5 AI Trader - Kaggle Phase 2A Candidate Search

Chay offline tren Kaggle: Discovery, Validation, Pre-live gate. Khong live/paper/demo tren Kaggle.
Ho tro thuat toan moi LightGBM/XGBoost khi bat INCLUDE_HEAVY_MODELS = True.
Default van an toan: chi ExtraTrees/RandomForest neu INCLUDE_HEAVY_MODELS = False.
Doc auto_improve_best.json roi kiem tra threshold stability tai best_threshold +/- 0.01.


In [ ]:
# Kaggle one-click safe candidate search: run this cell only
import os, sys, glob, json, shutil, subprocess, platform
from pathlib import Path

REPO_URL = "https://github.com/quoccuongphan300992-cmd/mt5-ai-trader.git"
REPO_DIR = Path("/kaggle/working/mt5-ai-trader")
OUTPUT_ROOT = Path("/kaggle/working/mt5_ai_outputs")
CANDIDATE_MODEL_DIR = OUTPUT_ROOT / "models" / "candidates"
LOCAL_CSV = Path("data/EURUSD_H1.csv")
SYMBOL = "EURUSD"
TIMEFRAME = "H1"
BARS = 50000
INCLUDE_HEAVY_MODELS = True  # True: add LightGBM/XGBoost; False: ExtraTrees/RandomForest only

GATES = {
    "discovery": {"max_rounds": 24, "grid_mode": "targeted", "folds": 5, "min_trades": 250, "min_profit_factor": 1.10, "min_expectancy": 0.03, "min_positive_fold_ratio": 0.8, "max_drawdown_limit": 0.25, "threshold_min": 0.48, "threshold_max": 0.65, "threshold_step": 0.01},
    "validation": {"max_rounds": 50, "grid_mode": "targeted", "folds": 5, "min_trades": 300, "min_profit_factor": 1.15, "min_expectancy": 0.05, "min_positive_fold_ratio": 1.0, "max_drawdown_limit": 0.20, "threshold_min": 0.48, "threshold_max": 0.65, "threshold_step": 0.01},
    "pre_live": {"max_rounds": 100, "grid_mode": "full", "folds": 5, "min_trades": 500, "min_profit_factor": 1.20, "min_expectancy": 0.08, "min_positive_fold_ratio": 1.0, "max_drawdown_limit": 0.15, "threshold_min": 0.50, "threshold_max": 0.68, "threshold_step": 0.01},
}
GATE_NAME = "discovery"  # discovery, validation, pre_live
GATE = GATES[GATE_NAME]

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Kaggle inputs:", glob.glob("/kaggle/input/*"))
print("Gate:", GATE_NAME, json.dumps(GATE, indent=2))
print("Include heavy models:", INCLUDE_HEAVY_MODELS)

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main"], check=True)

os.chdir(REPO_DIR)
subprocess.run(["git", "log", "--oneline", "-1"], check=True)
print("CWD:", os.getcwd())

req = Path("requirements-colab.txt") if Path("requirements-colab.txt").exists() else Path("requirements.txt")
safe_req = Path("/kaggle/working/requirements-kaggle.txt")
safe_lines = []
for line in req.read_text(encoding="utf-8").splitlines():
    stripped = line.strip()
    if not stripped or stripped.startswith("#"):
        continue
    if stripped.lower().startswith("metatrader5"):
        print("Skip Kaggle-incompatible dependency:", stripped)
        continue
    safe_lines.append(stripped)
safe_req.write_text("\n".join(safe_lines) + "\n", encoding="utf-8")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(safe_req)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ta"], check=True)

LOCAL_CSV.parent.mkdir(parents=True, exist_ok=True)
candidates = glob.glob("/kaggle/input/**/*.csv", recursive=True)
print("CSV candidates:")
for path in candidates[:50]:
    print(" -", path)
if len(candidates) > 50:
    print(f"... {len(candidates) - 50} more CSV files omitted")
if not candidates:
    raise FileNotFoundError("No CSV found under /kaggle/input. Add EURUSD H1 CSV dataset via Add Input.")
preferred = [x for x in candidates if "EURUSD" in os.path.basename(x).upper() and "H1" in os.path.basename(x).upper()]
source_csv = preferred[0] if preferred else candidates[0]
shutil.copy2(source_csv, LOCAL_CSV)
print("Using CSV:", source_csv)
print("LOCAL_CSV:", LOCAL_CSV.resolve(), LOCAL_CSV.stat().st_size, "bytes")

import pandas as pd
preview_df = pd.read_csv(LOCAL_CSV)
print("CSV shape:", preview_df.shape)
print("CSV columns:", preview_df.columns.tolist())
print(preview_df.head(3))

CANDIDATE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
os.environ["PYTHONUNBUFFERED"] = "1"
cmd = [
    sys.executable, "-u", "main.py", "auto-improve",
    "--csv", str(LOCAL_CSV),
    "--symbol", SYMBOL,
    "--timeframe", TIMEFRAME,
    "--bars", str(BARS),
    "--max-rounds", str(GATE["max_rounds"]),
    "--folds", str(GATE["folds"]),
    "--grid-mode", str(GATE.get("grid_mode", "smoke")),
    "--min-trades", str(GATE["min_trades"]),
    "--min-profit-factor", str(GATE["min_profit_factor"]),
    "--min-expectancy", str(GATE["min_expectancy"]),
    "--min-positive-fold-ratio", str(GATE["min_positive_fold_ratio"]),
    "--max-drawdown-limit", str(GATE["max_drawdown_limit"]),
    "--min", str(GATE["threshold_min"]),
    "--max", str(GATE["threshold_max"]),
    "--step", str(GATE["threshold_step"]),
    "--promotion-mode", "candidate-only",
    "--candidate-model-dir", str(CANDIDATE_MODEL_DIR),
]
if INCLUDE_HEAVY_MODELS:
    cmd.append("--include-heavy-models")
print("Running:", " ".join(cmd), flush=True)
subprocess.run(cmd, check=True, env=os.environ.copy())

print("=== CANDIDATE REPORT PREVIEW ===")
report_csv = REPO_DIR / "reports" / "auto_improve_candidates.csv"
if report_csv.exists():
    report_df = pd.read_csv(report_csv)
    inspect_cols = ["rank", "candidate_id", "direction", "model_type", "label_atr_tp_mult", "label_atr_sl_mult", "horizon", "filter_preset", "filter_applied", "filter_warning", "threshold", "trades", "profit_factor", "expectancy", "max_drawdown", "positive_fold_ratio", "candidate_pass", "fail_reasons", "score"]
    inspect_cols = [c for c in inspect_cols if c in report_df.columns]
    print(report_df[inspect_cols].head(30).to_string(index=False))
else:
    print("No report CSV found:", report_csv)

print("=== BEST CANDIDATE ===")
best_paths = [REPO_DIR / "reports" / "auto_improve_best.json", OUTPUT_ROOT / "reports" / "auto_improve_best.json"]
best_data = None
for best in best_paths:
    if best.exists():
        print(best)
        text = best.read_text(encoding="utf-8")
        print(text)
        best_data = json.loads(text)
        break
if best_data is None:
    print("No best file found. Check reports/auto_improve_fail_reasons.json.")

print("=== SAVED MODEL FILES ===")
model_files = sorted(CANDIDATE_MODEL_DIR.rglob("model.joblib"))
for path in model_files:
    print(path)
if not model_files:
    print("No model.joblib saved. No candidate passed gates.")

print("=== THRESHOLD STABILITY NEXT STEP ===")
print("If best candidate passes, rerun walk-forward at threshold -0.01, threshold, threshold +0.01 with same direction/model config.")
if best_data and best_data.get("best_candidate"):
    b = best_data["best_candidate"]
    direction = b.get("direction", "BUY")
    threshold = float(b.get("threshold", 0.53))
    label_method = b.get("label_method", "atr_path")
    tp = b.get("label_atr_tp_mult", 2.0)
    sl = b.get("label_atr_sl_mult", 1.0)
    horizon = b.get("horizon", 10)
    model_type = b.get("model_type", "extra_trees")
    filter_preset = b.get("filter_preset", "none")
    for t in [threshold - 0.01, threshold, threshold + 0.01]:
        print(
            f"!cd {REPO_DIR} && python main.py walk-forward --csv {LOCAL_CSV.resolve()} "
            f"--direction {direction} --folds 5 --threshold {t:.2f} "
            f"--model-type {model_type} --label-method {label_method} "
            f"--label-atr-tp-mult {tp} --label-atr-sl-mult {sl} --horizon {horizon} "
            f"--filter-preset {filter_preset}"
        )
else:
    print(f"!cd {REPO_DIR} && python main.py walk-forward --csv {LOCAL_CSV.resolve()} --direction BUY --folds 5 --threshold 0.52")
    print(f"!cd {REPO_DIR} && python main.py walk-forward --csv {LOCAL_CSV.resolve()} --direction BUY --folds 5 --threshold 0.53")
    print(f"!cd {REPO_DIR} && python main.py walk-forward --csv {LOCAL_CSV.resolve()} --direction BUY --folds 5 --threshold 0.54")

out_dir = Path("/kaggle/working/output")
out_dir.mkdir(parents=True, exist_ok=True)
for folder in ["models", "reports"]:
    dst_root = out_dir / folder
    dst_root.mkdir(parents=True, exist_ok=True)
    src_root = REPO_DIR / folder
    if src_root.exists():
        for item in src_root.glob("*"):
            dst = dst_root / item.name
            if item.is_dir():
                shutil.copytree(item, dst, dirs_exist_ok=True)
            else:
                shutil.copy2(item, dst)
if OUTPUT_ROOT.exists():
    shutil.copytree(OUTPUT_ROOT, out_dir / "mt5_ai_outputs", dirs_exist_ok=True)
zip_path = Path("/kaggle/working/mt5_ai_outputs_kaggle.zip")
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(str(zip_path)[:-4], "zip", out_dir)
print("=== OUTPUT FILES ===")
for path in sorted(out_dir.rglob("*")):
    if path.is_file():
        print(path)
print("ZIP:", zip_path, zip_path.stat().st_size, "bytes")
